In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import warnings
warnings.filterwarnings('ignore')

print("1. Loading NAV data from Database...")
# Connect to the SQLite database
conn = sqlite3.connect('../data/db/bluestock_mf.db')

# Fetch the NAV data, ordered chronologically per fund (Crucial for pct_change)
query = """
SELECT f.amfi_code, d.scheme_name, f.date, f.nav
FROM fact_nav f
JOIN dim_fund d ON f.amfi_code = d.amfi_code
ORDER BY f.amfi_code, f.date
"""
df_nav = pd.read_sql_query(query, conn)
conn.close()

print("2. Computing Daily Returns...")
df_nav['date'] = pd.to_datetime(df_nav['date'])

# Calculate daily return: (nav_t / nav_t-1) - 1
# Using groupby ensures the calculation doesn't accidentally bleed across different funds
df_nav['daily_return'] = df_nav.groupby('amfi_code')['nav'].pct_change()

print("3. Computing Overall Annualized Returns...")
# Formula: (1 + daily_return).prod()^(252/n) - 1
def compute_annualized(group):
    # Drop NaNs (like the very first day of trading which has no previous day)
    returns = group['daily_return'].dropna()
    n = len(returns)
    
    # Require at least 6 months (approx 126 trading days) of data for a valid metric
    if n < 126:  
        return np.nan
        
    compounded = np.prod(1 + returns)
    return (compounded ** (252 / n)) - 1

# Apply the custom formula to every fund
df_annualized = df_nav.groupby(['amfi_code', 'scheme_name']).apply(compute_annualized).reset_index()
df_annualized.columns = ['amfi_code', 'scheme_name', 'annualized_return']

# Convert decimal to percentage for readability (e.g., 0.15 -> 15.0%)
df_annualized['annualized_return_pct'] = (df_annualized['annualized_return'] * 100).round(2)

print("4. Saving Results...")
os.makedirs('../data/processed', exist_ok=True)
output_path = '../data/processed/returns_computed.csv'

# Save the annualized returns as requested by the deliverable
df_annualized.to_csv(output_path, index=False)

# We will also keep df_nav (with daily returns) in memory for Tasks 3, 4, 5, and 6!
print(f"✅ Task 1 Complete! Annualized returns saved to {output_path}")

# Preview the top performers
df_annualized.sort_values(by='annualized_return', ascending=False).head()

1. Loading NAV data from Database...
2. Computing Daily Returns...
3. Computing Overall Annualized Returns...
4. Saving Results...
✅ Task 1 Complete! Annualized returns saved to ../data/processed/returns_computed.csv


,amfi_code,scheme_name,annualized_return,annualized_return_pct
25,120505,ICICI Pru Midcap Fund - Regular - Growth,0.216357,21.64
21,119598,SBI Small Cap Fund - Regular Plan - Growth,0.213804,21.38
39,149324,DSP Small Cap Fund - Regular - Growth,0.212943,21.29
36,148569,Mirae Asset Tax Saver Fund - Regular - Growth,0.210804,21.08
34,148567,Mirae Asset Large Cap Fund - Regular - Growth,0.204622,20.46


In [2]:
import pandas as pd
import numpy as np
import os

print("1. Preparing data for CAGR calculation...")
# Find the global maximum date in the dataset to act as our "Present Day"
max_date = df_nav['date'].max()
print(f"Latest trading date in dataset: {max_date.date()}")

# Define our historical cutoffs (1 Year, 3 Years, 5 Years ago)
timeframes = {
    '1Y': max_date - pd.DateOffset(years=1),
    '3Y': max_date - pd.DateOffset(years=3),
    '5Y': max_date - pd.DateOffset(years=5)
}

cagr_results = []

print("2. Calculating Compound Annual Growth Rates (CAGR)...")
# Loop through every single mutual fund individually
for amfi, group in df_nav.groupby('amfi_code'):
    scheme_name = group['scheme_name'].iloc[0]
    
    # Ensure data is sorted chronologically
    group = group.sort_values('date')
    
    # Get the ending NAV (the absolute latest price for this fund)
    nav_end = group['nav'].iloc[-1]
    end_date = group['date'].iloc[-1]
    
    fund_data = {'amfi_code': amfi, 'scheme_name': scheme_name}
    
    # Calculate for 1Y, 3Y, and 5Y
    for label, cutoff_date in timeframes.items():
        # Find the closest trading date on or AFTER our exact cutoff date
        past_data = group[group['date'] >= cutoff_date]
        
        if past_data.empty:
            fund_data[f'cagr_{label}_pct'] = np.nan
            continue
            
        nav_start = past_data['nav'].iloc[0]
        start_date = past_data['date'].iloc[0]
        
        # Calculate the exact number of years between start and end (handles leap years)
        days_diff = (end_date - start_date).days
        years_diff = days_diff / 365.25
        
        # Validation: If a fund is only 2 years old, it shouldn't get a 3Y or 5Y CAGR
        target_years = int(label[0])
        if years_diff < (target_years * 0.9): # Must have at least 90% of the required time
            fund_data[f'cagr_{label}_pct'] = np.nan
        else:
            # The core CAGR Formula
            cagr = (nav_end / nav_start) ** (1 / years_diff) - 1
            fund_data[f'cagr_{label}_pct'] = round(cagr * 100, 2)
            
    cagr_results.append(fund_data)

# Combine all the individual fund results into one master table
df_cagr = pd.DataFrame(cagr_results)

print("3. Saving CAGR Report...")
output_path = '../data/processed/cagr_report.csv'
df_cagr.to_csv(output_path, index=False)
print(f"✅ Task 2 Complete! CAGR report successfully saved to {output_path}")

# Preview the calculated metrics
df_cagr.head()

1. Preparing data for CAGR calculation...
Latest trading date in dataset: 2026-05-29
2. Calculating Compound Annual Growth Rates (CAGR)...
3. Saving CAGR Report...
✅ Task 2 Complete! CAGR report successfully saved to ../data/processed/cagr_report.csv


,amfi_code,scheme_name,cagr_1Y_pct,cagr_3Y_pct,cagr_5Y_pct
0,100016,HDFC Top 100 Fund - Regular Plan - Growth,-2.23,1.29,NaN
1,100025,HDFC Short Term Debt Fund - Regular - Growth,3.71,3.92,NaN
2,100033,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,53.28,32.43,NaN
3,101206,ABSL Frontline Equity Fund - Regular - Growth,47.96,28.96,NaN
4,101207,ABSL Small Cap Fund - Regular - Growth,-24.00,-4.15,NaN


In [3]:
import pandas as pd
import numpy as np
import os

print("1. Preparing data for Sharpe Ratio calculation...")
# Set the risk-free rate to 6.5% as per the rubric
rf_rate = 0.065 

def compute_sharpe(group):
    # Extract the daily returns we calculated in Task 1
    returns = group['daily_return'].dropna()
    n = len(returns)
    
    # Require at least 6 months of data, and avoid dividing by zero if a fund has no volatility
    if n < 126 or returns.std() == 0:
        return np.nan
        
    # 1. Calculate the Annualized Return (Rp)
    compounded = np.prod(1 + returns)
    rp = (compounded ** (252 / n)) - 1
    
    # 2. Calculate Annualized Volatility (Standard Deviation * sqrt(252))
    volatility = returns.std() * np.sqrt(252)
    
    # 3. Compute the Sharpe Ratio Formula: (Rp - Rf) / Volatility
    sharpe = (rp - rf_rate) / volatility
    
    return round(sharpe, 2)

print("2. Computing Sharpe Ratios...")
# Apply the custom function to our grouped NAV data
df_sharpe = df_nav.groupby(['amfi_code', 'scheme_name']).apply(compute_sharpe).reset_index()
df_sharpe.columns = ['amfi_code', 'scheme_name', 'sharpe_ratio']

# Let's sort to see who the top performers are
df_sharpe = df_sharpe.sort_values(by='sharpe_ratio', ascending=False)

print("3. Saving Sharpe Ratio Results...")
output_path = '../data/processed/sharpe_values.csv'
df_sharpe.to_csv(output_path, index=False)
print(f"✅ Task 3 Complete! Sharpe ratios successfully saved to {output_path}")

# Preview the top 5 funds by risk-adjusted return
df_sharpe.head()

1. Preparing data for Sharpe Ratio calculation...
2. Computing Sharpe Ratios...
3. Saving Sharpe Ratio Results...
✅ Task 3 Complete! Sharpe ratios successfully saved to ../data/processed/sharpe_values.csv


,amfi_code,scheme_name,sharpe_ratio
34,148567,Mirae Asset Large Cap Fund - Regular - Growth,1.16
30,120843,Kotak Flexicap Fund - Regular - Growth,1.03
36,148569,Mirae Asset Tax Saver Fund - Regular - Growth,0.97
25,120505,ICICI Pru Midcap Fund - Regular - Growth,0.93
19,119551,SBI Bluechip Fund - Regular Plan - Growth,0.92


In [4]:
import pandas as pd
import numpy as np
import os

print("1. Preparing data for Sortino Ratio calculation...")
rf_rate = 0.065 

def compute_sortino(group):
    # Extract the daily returns
    returns = group['daily_return'].dropna()
    n = len(returns)
    
    if n < 126:
        return np.nan
        
    # 1. Calculate the Annualized Return (Rp)
    compounded = np.prod(1 + returns)
    rp = (compounded ** (252 / n)) - 1
    
    # 2. ISOLATE DOWNSIDE RISK: Filter for only negative return days
    negative_returns = returns[returns < 0]
    
    # If the fund magically never had a down day, we can't divide by zero
    if len(negative_returns) == 0 or negative_returns.std() == 0:
        return np.nan
        
    # 3. Calculate Annualized Downside Volatility
    downside_std = negative_returns.std() * np.sqrt(252)
    
    # 4. Compute Sortino Ratio Formula: (Rp - Rf) / Downside_Std
    sortino = (rp - rf_rate) / downside_std
    
    return round(sortino, 2)

print("2. Computing Sortino Ratios...")
# Apply the custom function
df_sortino = df_nav.groupby(['amfi_code', 'scheme_name']).apply(compute_sortino).reset_index()
df_sortino.columns = ['amfi_code', 'scheme_name', 'sortino_ratio']

# Sort to see the top performers
df_sortino = df_sortino.sort_values(by='sortino_ratio', ascending=False)

print("3. Saving Sortino Ratio Results...")
output_path = '../data/processed/sortino_values.csv'
df_sortino.to_csv(output_path, index=False)
print(f"✅ Task 4 Complete! Sortino ratios successfully saved to {output_path}")

# Preview the top 5 funds by downside risk-adjusted return
df_sortino.head()

1. Preparing data for Sortino Ratio calculation...
2. Computing Sortino Ratios...
3. Saving Sortino Ratio Results...
✅ Task 4 Complete! Sortino ratios successfully saved to ../data/processed/sortino_values.csv


,amfi_code,scheme_name,sortino_ratio
34,148567,Mirae Asset Large Cap Fund - Regular - Growth,1.62
30,120843,Kotak Flexicap Fund - Regular - Growth,1.59
36,148569,Mirae Asset Tax Saver Fund - Regular - Growth,1.43
19,119551,SBI Bluechip Fund - Regular Plan - Growth,1.37
25,120505,ICICI Pru Midcap Fund - Regular - Growth,1.35


In [5]:
import pandas as pd
import numpy as np
from scipy import stats
import os

print("1. Loading Benchmark (Nifty 100) data...")
# Dynamically find the benchmark file in your raw data folder
raw_files = os.listdir('../data/raw')
bench_file = [f for f in raw_files if 'nifty' in f.lower() or 'benchmark' in f.lower()][0]
df_bench = pd.read_csv(f'../data/raw/{bench_file}')

# Clean columns dynamically
df_bench.columns = df_bench.columns.str.lower().str.strip().str.replace(' ', '_')
date_col = [col for col in df_bench.columns if 'date' in col][0]
price_col = [col for col in df_bench.columns if 'close' in col or 'nav' in col or 'price' in col or 'value' in col][0]

# Format dates and calculate the Nifty 100 daily returns
df_bench[date_col] = pd.to_datetime(df_bench[date_col])
df_bench = df_bench.sort_values(by=date_col)
df_bench['bench_return'] = df_bench[price_col].pct_change()

print("2. Merging Fund Returns with Benchmark Returns...")
# Merge with the df_nav we kept in memory from Task 1
df_merged = pd.merge(df_nav, df_bench[[date_col, 'bench_return']], left_on='date', right_on=date_col, how='inner')
df_merged = df_merged.dropna(subset=['daily_return', 'bench_return'])

print("3. Computing Alpha and Beta (OLS Regression)...")
alpha_beta_results = []

for amfi, group in df_merged.groupby('amfi_code'):
    scheme_name = group['scheme_name'].iloc[0]
    
    # We need at least 6 months of overlapping data for a statistically significant regression
    if len(group) < 126:
        continue
        
    # x = Market Returns (Nifty 100), y = Fund Returns
    x = group['bench_return']
    y = group['daily_return']
    
    # Run the Linear Regression
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
    
    # Beta is the slope. Alpha is the intercept (annualized and converted to percentage)
    beta = round(slope, 2)
    alpha = round(intercept * 252 * 100, 2) 
    
    alpha_beta_results.append({
        'amfi_code': amfi,
        'scheme_name': scheme_name,
        'alpha_pct': alpha,
        'beta': beta,
        'r_squared': round(r_value**2, 2) # Added R-Squared to show how closely it tracks the index!
    })

df_alpha_beta = pd.DataFrame(alpha_beta_results)
df_alpha_beta = df_alpha_beta.sort_values(by='alpha_pct', ascending=False)

print("4. Saving Alpha & Beta Results...")
output_path = '../data/processed/alpha_beta.csv'
df_alpha_beta.to_csv(output_path, index=False)
print(f"✅ Task 5 Complete! Alpha/Beta successfully saved to {output_path}")

# Preview the top 5 funds by pure Manager Skill (Alpha)
df_alpha_beta.head()

1. Loading Benchmark (Nifty 100) data...
2. Merging Fund Returns with Benchmark Returns...
3. Computing Alpha and Beta (OLS Regression)...
4. Saving Alpha & Beta Results...
✅ Task 5 Complete! Alpha/Beta successfully saved to ../data/processed/alpha_beta.csv


,amfi_code,scheme_name,alpha_pct,beta,r_squared
39,149324,DSP Small Cap Fund - Regular - Growth,29.60,0.0,0.0
36,148569,Mirae Asset Tax Saver Fund - Regular - Growth,29.59,-0.0,0.0
21,119598,SBI Small Cap Fund - Regular Plan - Growth,29.51,0.0,0.0
34,148567,Mirae Asset Large Cap Fund - Regular - Growth,27.89,-0.0,0.0
25,120505,ICICI Pru Midcap Fund - Regular - Growth,27.82,0.0,0.0


In [6]:
import pandas as pd
import numpy as np
import os

print("1. Preparing data for Maximum Drawdown calculation...")

def compute_max_drawdown(group):
    # Ensure chronological order
    group = group.sort_values('date')
    
    # Calculate the running maximum (highest NAV seen so far up to each date)
    group['running_max'] = group['nav'].cummax()
    
    # Calculate current drawdown: (Current NAV / Running Max) - 1
    group['drawdown'] = (group['nav'] / group['running_max']) - 1
    
    # The Maximum Drawdown is the lowest value in the drawdown column
    max_dd = group['drawdown'].min()
    
    # Handle rare cases where a fund perfectly only goes up
    if max_dd >= 0 or pd.isna(max_dd):
        return pd.Series({
            'max_drawdown_pct': 0.0,
            'peak_date': None,
            'trough_date': None
        })
        
    # Find the exact date the worst drop happened (the absolute bottom / trough)
    trough_idx = group['drawdown'].idxmin()
    trough_date = group.loc[trough_idx, 'date']
    
    # Find the peak date BEFORE that trough
    # We slice the data up to the trough and find the highest NAV
    historical_data = group.loc[:trough_idx]
    peak_idx = historical_data['nav'].idxmax()
    peak_date = historical_data.loc[peak_idx, 'date']
    
    return pd.Series({
        'max_drawdown_pct': round(max_dd * 100, 2),
        'peak_date': peak_date.strftime('%Y-%m-%d'),
        'trough_date': trough_date.strftime('%Y-%m-%d')
    })

print("2. Computing Maximum Drawdowns and Worst Periods...")
# Apply the custom function
df_drawdown = df_nav.groupby(['amfi_code', 'scheme_name']).apply(compute_max_drawdown).reset_index()

# Sort from worst (most negative) to best to see who crashed the hardest
df_drawdown = df_drawdown.sort_values(by='max_drawdown_pct')

print("3. Saving Maximum Drawdown Results...")
output_path = '../data/processed/max_drawdown.csv'
df_drawdown.to_csv(output_path, index=False)
print(f"✅ Task 6 Complete! Max drawdowns successfully saved to {output_path}")

# Preview the worst 5 drawdowns (The most painful drops)
df_drawdown.head()

1. Preparing data for Maximum Drawdown calculation...
2. Computing Maximum Drawdowns and Worst Periods...
3. Saving Maximum Drawdown Results...
✅ Task 6 Complete! Max drawdowns successfully saved to ../data/processed/max_drawdown.csv


,amfi_code,scheme_name,max_drawdown_pct,peak_date,trough_date
22,119599,SBI Small Cap Fund - Direct Plan - Growth,-52.57,2023-01-17,2025-10-28
17,119095,Axis Small Cap Fund - Regular - Growth,-51.68,2025-05-22,2026-05-11
4,101207,ABSL Small Cap Fund - Regular - Growth,-35.45,2024-11-21,2026-05-11
39,149324,DSP Small Cap Fund - Regular - Growth,-31.17,2024-05-03,2025-01-03
21,119598,SBI Small Cap Fund - Regular Plan - Growth,-28.71,2024-08-28,2025-05-14
